## 01. Import dependancies

In [1]:
import warnings
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_auc_score
)

warnings.filterwarnings('ignore')


## Load dataset

In [2]:
X_train=pd.read_csv('artifacts/X_train.csv')
X_test=pd.read_csv('artifacts/X_test.csv')
y_train = pd.read_csv('artifacts/y_train.csv').squeeze("columns")
y_test = pd.read_csv('artifacts/y_test.csv').squeeze("columns")

In [3]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("Train lengths:", len(X_train), len(y_train))
print("Test lengths:", len(X_test), len(y_test))

X_train: (5625, 45)
y_train: (5625,)
X_test: (1407, 45)
y_test: (1407,)
Train lengths: 5625 5625
Test lengths: 1407 1407


## 03. Model training

### defining parameters


In [4]:
lr_param_grid = {
    'model__max_iter': [1000, 5000],
    'model__C': [0.1, 1.0, 10.0]
}

dt_param_grid = {
    'model__max_depth': [5, 8, 10],
    'model__min_samples_leaf': [2, 5, 10],
    'model__criterion': ['gini', 'entropy']
}

rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 10, 12],
    'model__min_samples_leaf': [2, 5, 10],
    'model__class_weight': ['balanced', 'balanced_subsample', None]
}

param_grids = {
    'Logistic Regression': lr_param_grid,
    'Decision Tree': dt_param_grid,
    'Random Forest': rf_param_grid
}

### Define models

In [5]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
}

### Configure K-fold CV

In [6]:
cv = StratifiedKFold(
    n_splits=6,
    random_state=42,
    shuffle=True
)

### Model Training

In [7]:
grid_search_results = {}

for model_name, model in models.items():
    print(f'\n--- Tuning {model_name} ---')

    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    param_grid = param_grids[model_name]

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='f1',
        verbose=1,
        return_train_score=False,
        n_jobs=-1
    )

    print(f"Fitting GridSearchCV for {model_name}")

    grid_search.fit(X_train, y_train)

    grid_search_results[model_name] = grid_search

    print(f"{model_name} GridSearchCV completed...")
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")


--- Tuning Logistic Regression ---
Fitting GridSearchCV for Logistic Regression
Fitting 6 folds for each of 6 candidates, totalling 36 fits
Logistic Regression GridSearchCV completed...
Best parameters: {'model__C': 1.0, 'model__max_iter': 1000}
Best CV score: 0.6241

--- Tuning Decision Tree ---
Fitting GridSearchCV for Decision Tree
Fitting 6 folds for each of 18 candidates, totalling 108 fits
Decision Tree GridSearchCV completed...
Best parameters: {'model__criterion': 'entropy', 'model__max_depth': 8, 'model__min_samples_leaf': 2}
Best CV score: 0.6019

--- Tuning Random Forest ---
Fitting GridSearchCV for Random Forest
Fitting 6 folds for each of 54 candidates, totalling 324 fits
Random Forest GridSearchCV completed...
Best parameters: {'model__class_weight': 'balanced_subsample', 'model__max_depth': 10, 'model__min_samples_leaf': 10, 'model__n_estimators': 100}
Best CV score: 0.6316


In [9]:
for model_name, grid in grid_search_results.items():
    # Get the best estimator from GridSearchCV
    best_model = grid.best_estimator_

    # Make predictions on the test set
    y_pred = best_model.predict(X_test)

    # Get probability of Churn
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]

    # Compute metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    rf_auc = roc_auc_score(y_test, y_pred_proba)

    print(f"\n--- {model_name} Test Set Metrics ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"ROC-score: {rf_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


--- Logistic Regression Test Set Metrics ---
Accuracy: 0.7356
Precision: 0.5018
Recall: 0.7594
F1-score: 0.6043
ROC-score: 0.8300

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.73      0.80      1033
           1       0.50      0.76      0.60       374

    accuracy                           0.74      1407
   macro avg       0.70      0.74      0.70      1407
weighted avg       0.79      0.74      0.75      1407


--- Decision Tree Test Set Metrics ---
Accuracy: 0.7562
Precision: 0.5307
Recall: 0.7166
F1-score: 0.6098
ROC-score: 0.8017

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1033
           1       0.53      0.72      0.61       374

    accuracy                           0.76      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79      0.76      0.77      1407


--- Random Forest Test Set Metrics ---
Ac